# 1. Episodes of care, flags, and CT reports for the NLP

**What it does**
1. Takes every encounter in the extraction (adults with an SDH code in the ER or inpatient record).
2. Joins encounters of the same patient into one **episode of care** when GEMINI's `epicare` links them
   or the next admission starts within 12 h of the last discharge (a transfer).
3. **Flags** each episode instead of excluding it: SDH coded only in the ER, SDH that developed in
   hospital, elective admission, transferred in from outside GEMINI, no CT report, outside 2015-2024.
4. Collects **every head CT report** of every episode (not just the first: the NLP has to find the
   first CT that actually shows the SDH).

**Output** (`patient_level/`): `episodes.csv`, `episode_encounters.csv`, `ct_reports_for_nlp.csv`,
`ct_reports_index.csv`. Aggregate: `step1_counts.csv` and four check tables.

**Time:** a few minutes (reads the radiology table).

> **Privacy:** everything this notebook *displays* comes from `aggregate/` (counts with cells under 6 suppressed). Files in `patient_level/` stay on GEMINI: don't print their rows, and never paste them outside GEMINI.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# To write the output somewhere else, uncomment and edit BEFORE importing sdh_config:
# os.environ["SDH_OUTPUT_DIR"] = str(Path.home() / "sdh_phenotype_output")

PHENO = Path.cwd().parent              # the phenotype/ folder (this notebook is in phenotype/notebooks/)
sys.path.insert(0, str(PHENO))
import sdh_config as cfg
import sdh_io as io
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 220)
AGG = cfg.AGGREGATE_DIR


def show(name, rows=None, note=None):
    """Display one aggregate (cell-suppressed) table from the output folder."""
    path = AGG / name
    if not path.exists():
        print(f"{name}: not there yet (run the step above first)")
        return None
    df = pd.read_csv(path)
    display(Markdown(f"**{name}**" + (f" - {note}" if note else "")))
    display(df.head(rows) if rows else df)
    return df


def share(names, rows=80):
    """Print aggregate tables as text to paste to Claude / your team (counts only, small cells suppressed)."""
    for name in names:
        path = AGG / name
        if path.exists():
            df = pd.read_csv(path)
            print(f"===== {name} ({len(df)} rows{', first ' + str(rows) if len(df) > rows else ''}) =====")
            print(df.head(rows).to_csv(index=False))
        else:
            print(f"===== {name}: not found =====")

## Run

In [ ]:
%run ../01_episodes_and_ct_reports.py

## Results

In [ ]:
show("step1_counts.csv", note="patients and episodes carrying each flag; the last row is how many reports the NLP will read");

**Look for:** episodes vs patients (how many patients have more than one SDH episode); how many are
in-hospital onset and ER-only; the share with **no CT report** (about 28% expected).

In [ ]:
show("check_transfer_linking.csv", note="how transfers were joined");

**Look for:** most joins found by *both* epicare and the 12 h rule. Many 'epicare only' joins would mean the 12 h window is too short; many 'time gap only' that it is too long.

In [ ]:
cov = show("check_ct_coverage_by_hospital_year.csv", note="episodes with a CT report, per hospital and year")
if cov is not None:
    display(Markdown("**% of episodes with a CT report** (blank = small cell)"))
    display(cov.pivot_table(index=["hospital_num", "hospital_type"], columns="year", values="pct_with_ct_report"))

**Look for:** hospitals or years with low or no CT reports. Those patients would drop out of the NLP cohort, so check whether the missing reports are spread out or concentrated.

In [ ]:
show("check_ct_names.csv", rows=40, note="CT names counted as head CTs (excluded_name = True were left out)");

**Look for:** head CTs wrongly excluded, or non-head scans (CTA, spine) wrongly included.

## Share

In [ ]:
share(["step1_counts.csv", "check_transfer_linking.csv", "check_ct_coverage_by_hospital_year.csv", "check_ct_names.csv"])